In [4]:
import numpy as np
import gymnasium as gym
from collections import defaultdict

In [5]:
class SARSALambdaAgent:
    def __init__(self, env, alpha=0.5, gamma=0.99, epsilon=0.5, lam=0.9):
        self.env = env
        self.alpha = alpha        # learning rate
        self.gamma = gamma        # discount factor
        self.epsilon = epsilon    # exploration rate
        self.lam = lam            # trace decay rate

        self.Q = defaultdict(lambda: np.zeros(env.action_space.n))

    def choose_action(self, state):
        if np.random.rand() < self.epsilon:
            return self.env.action_space.sample()
        else:
            return np.argmax(self.Q[state])

    def learn(self, episodes=10000):
        for ep in range(episodes):
            state = self.env.reset()[0]
            action = self.choose_action(state)

            # Initialize eligibility traces
            E = defaultdict(lambda: np.zeros(self.env.action_space.n))
            done = False

            while not done:
                next_state, reward, done, _, info = self.env.step(action)
                next_action = self.choose_action(next_state)

                td_target = reward + self.gamma * self.Q[next_state][next_action]
                td_error = td_target - self.Q[state][action]

                # Update eligibility trace for current (state, action)
                E[state][action] += 1

                # Update Q-values and traces
                for s in E:
                    for a in range(self.env.action_space.n):
                        self.Q[s][a] += self.alpha * td_error * E[s][a]
                        E[s][a] *= self.gamma * self.lam  # decay

                state, action = next_state, next_action

            if (ep + 1) % 1000 == 0:
                print(f"Episode {ep + 1} completed")

    def get_policy(self):
        policy = np.zeros(self.env.observation_space.n, dtype=int)
        for state in range(self.env.observation_space.n):
            if state in self.Q:
                policy[state] = np.argmax(self.Q[state])
        return policy




In [2]:
def print_policy(policy, shape=(4, 12)):
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    grid = np.array([arrows.get(a, ' ') for a in policy]).reshape(shape)
    grid[3, 0] = 'S'  # Start
    grid[3, 11] = 'G'  # Goal
    for i in range(1, 11):
        grid[3, i] = 'C'  # Cliff
    for row in grid:
        print(" ".join(row))

In [6]:
env = gym.make("CliffWalking-v0")
agent = SARSALambdaAgent(env, alpha=0.5, gamma=0.99, epsilon=0.1, lam=0.9)
agent.learn(episodes=10000)
policy = agent.get_policy()
print_policy(policy)

Episode 1000 completed
Episode 2000 completed
Episode 3000 completed
Episode 4000 completed
Episode 5000 completed
Episode 6000 completed
Episode 7000 completed
Episode 8000 completed
Episode 9000 completed
Episode 10000 completed
← → → ↑ → ↓ → ↓ → ↓ → ↓
↑ → ← → ↑ → → → → → ↑ ↓
→ → → → → ↑ ↑ ↑ → ↑ → ↓
S C C C C C C C C C C G
